In [0]:
%sql

CREATE OR REPLACE TABLE currency.`03_gold_currency`.fct_quotes
USING DELTA
AS
  
-- Minimum quote date per currency
WITH base AS (
SELECT
  MIN(q.dt_period) AS min_date,
  q.dc_currency
FROM currency.`02_silver_currency`.quotes q 
GROUP BY q.dc_currency
),

-- Generate all dates in the period
dates_period AS (
SELECT
    dc_currency,
    explode(
        sequence(
            min_date,
            current_date(),
            interval 1 day
        )
    ) AS dt_period
FROM base
),

-- Join the quotes with the full date range
full_data AS (
SELECT
    dp.dt_period,
    dp.dc_currency,
    COALESCE(q.vl_quote, NULL) AS vl_quote
FROM dates_period dp
LEFT JOIN currency.`02_silver_currency`.quotes q 
ON dp.dc_currency = q.dc_currency 
    AND dp.dt_period = q.dt_period
),

-- Fill the gaps with the last known quote of each currency
fct_quotes_final AS (
SELECT
    dt_period,
    dc_currency,
    last_value(vl_quote, true)
    OVER (
        PARTITION BY dc_currency
        ORDER BY dt_period
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS vl_quote
FROM full_data
)
SELECT * FROM fct_quotes_final
ORDER BY dt_period, dc_currency

In [0]:
%sql
SELECT * FROM currency.`03_gold_currency`.fct_quotes
ORDER BY dt_period, dc_currency